In [1]:
import os

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASETS = ["Cora", "PubMed"]

print("BASE_DIR =", BASE_DIR)
print("DATA_DIR =", DATA_DIR)
print("RESULTS_DIR =", RESULTS_DIR)
print("DATASETS =", DATASETS)


BASE_DIR = /home/amir/Desktop/proposal
DATA_DIR = /home/amir/Desktop/proposal/data
RESULTS_DIR = /home/amir/Desktop/proposal/results
DATASETS = ['Cora', 'PubMed']


In [2]:
import os, sys, json, time, platform
import torch, torch_geometric, sklearn, networkx, matplotlib, pandas

env = {
    "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "sklearn": sklearn.__version__,
    "networkx": networkx.__version__,
    "matplotlib": matplotlib.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}

print(json.dumps(env, indent=2))

with open(os.path.join(RESULTS_DIR, "env_info.json"), "w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)

print("Saved:", os.path.join(RESULTS_DIR, "env_info.json"))


{
  "timestamp_utc": "2026-01-28 10:01:28",
  "python": "3.12.7",
  "platform": "Linux-6.11.0-29-generic-x86_64-with-glibc2.40",
  "torch": "2.10.0+cpu",
  "torch_geometric": "2.7.0",
  "sklearn": "1.8.0",
  "networkx": "3.6.1",
  "matplotlib": "3.10.8",
  "cuda_available": false,
  "device_name": "cpu"
}
Saved: /home/amir/Desktop/proposal/results/env_info.json


In [3]:
import random
import numpy as np
import torch

SEED = 12
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

CFG = {
    "epochs": 200,
    "lr": 0.01,
    "weight_decay": 5e-4,
    "hidden_dim": 64,
    "dropout": 0.5,
}
print(CFG)


DEVICE: cpu
{'epochs': 200, 'lr': 0.01, 'weight_decay': 0.0005, 'hidden_dim': 64, 'dropout': 0.5}


<div dir="rtl" style="text-align:right; font-family: Vazirmatn, IRANSans, Tahoma, Arial; line-height:1.9;">

## دلیل انتخاب هایپرپارامترها در این پروژه

در این پروژه هایپرپارامترها را طوری انتخاب کردیم که برای دیتاست‌های خانواده‌ی **Planetoid** (مانند CiteSeer، Cora و PubMed) تنظیماتی «پایه، پایدار و قابل دفاع» باشد. از آن‌جا که هدف اصلی ما مقایسه‌ی دو معماری **GCN** و **GraphSAGE** روی دو دیتاست **Cora** و **PubMed** است، تلاش کردیم تنظیمات را تا حد ممکن **ثابت و مشترک** نگه داریم تا تفاوت نتایج، بیشتر ناشی از تفاوت مدل‌ها باشد نه ناشی از دست‌کاری جداگانه‌ی هایپرپارامترها برای هر دیتاست یا هر مدل. همچنین با توجه به اجرای پروژه روی CPU، تنظیمات به شکلی انتخاب شده که هم قابل اجرا باشد و هم به نتیجه‌ی قابل قبول برسد.

### چرا دو لایه (Two-layer) استفاده کردیم؟

مسئله‌ی ما **طبقه‌بندی گره‌ها** است و مدل باید از ساختار گراف (همسایه‌ها) اطلاعات بگیرد. با دو لایه پیام‌رسانی، هر گره می‌تواند اطلاعات همسایه‌های یک‌پرش و دوپرش خودش را در نمایش نهایی وارد کند. در گراف‌های استنادی Planetoid معمولاً همین محدوده‌ی محلی برای یادگیری سیگنال کلاس‌ها کافی است. از طرفی اگر تعداد لایه‌ها زیاد شود، مشکل شناخته‌شده‌ی **Over-smoothing** اتفاق می‌افتد؛ یعنی نمایش گره‌ها بیش از حد به هم شبیه می‌شود و مدل قدرت تفکیک کلاس‌ها را از دست می‌دهد. بنابراین دو لایه یک انتخاب متعادل است: استفاده‌ی کافی از ساختار گراف، بدون افزایش ریسک over-smoothing و بدون سنگین شدن محاسبات.

### چرا hidden_dim = 64 انتخاب شد؟

بعد پنهان ۶۴ یک اندازه‌ی متوسط و مطمئن است. اگر hidden_dim خیلی کوچک باشد، مدل ظرفیت کافی برای یادگیری ترکیبِ ویژگی‌های گره‌ها و سیگنال ساختاری را ندارد و **underfitting** رخ می‌دهد. اگر خیلی بزرگ باشد، تعداد پارامترها افزایش پیدا می‌کند و مخصوصاً در دیتاست‌هایی با برچسب‌های آموزشی محدود (مانند Cora)، احتمال **overfitting** بالا می‌رود و زمان آموزش نیز بیشتر می‌شود. مقدار ۶۴ معمولاً یک تعادل مناسب بین ظرفیت مدل و ریسک overfitting ایجاد می‌کند و برای اجرای CPU هم انتخاب عملی‌تری است.

### چرا dropout = 0.5 گذاشتیم؟

در این دیتاست‌ها تعداد نمونه‌های برچسب‌دار برای آموزش محدود است و مدل‌های گرافی هم به دلیل بهره‌گیری از همسایه‌ها ممکن است سریع‌تر overfit شوند. Dropout با خاموش کردن تصادفی بخشی از نورون‌ها در زمان آموزش، مدل را وادار می‌کند به یک مسیر خاص وابسته نشود و تعمیم بهتری داشته باشد. مقدار ۰٫۵ یک regularization نسبتاً قوی است و معمولاً در شرایط داده‌ی آموزشی محدود به پایداری آموزش و عملکرد بهتر روی validation کمک می‌کند.

### چرا learning rate = 0.01 انتخاب شد؟

برای بهینه‌ساز Adam در تنظیمات پایه‌ی بسیاری از مدل‌های GNN روی دیتاست‌های Planetoid، مقدار ۰٫۰۱ یک انتخاب رایج است: نه آن‌قدر بزرگ که آموزش ناپایدار شود، و نه آن‌قدر کوچک که یادگیری خیلی کند پیش برود. هدف ما این بوده که آموزش هم **سریع** باشد و هم **پایدار** تا بتوان نتایج دو مدل را منصفانه مقایسه کرد.

### چرا weight_decay = 5e-4 انتخاب شد؟

Weight decay نقش regularization دارد و جلوی بزرگ شدن بیش از حد وزن‌ها را می‌گیرد. چون تعداد epoch نسبتاً زیاد است و داده‌ی آموزشی محدود، بدون regularization مدل ممکن است روی train حفظی شود. مقدار 5e-4 یک مقدار رایج در تنظیمات پایه‌ی GCN برای دیتاست‌های Planetoid است و معمولاً باعث بهتر شدن generalization می‌شود بدون اینکه یادگیری را بیش از حد محدود کند.

### چرا epochs = 200 انتخاب شد؟

در دیتاست‌های Planetoid معمولاً مدل در ده‌ها epoch اول روند اصلی یادگیری را طی می‌کند، اما برای اینکه مدل فرصت کافی برای همگرایی داشته باشد و نتیجه‌ی قابل مقایسه‌ای تولید کند، تعداد epoch را ۲۰۰ گذاشتیم. علاوه بر این، چون ما **بهترین مدل بر اساس validation** را نگه می‌داریم (best_val_acc)، قرار نیست فقط خروجی epoch آخر را گزارش کنیم؛ بلکه این تعداد epoch کمک می‌کند شانس پیدا کردن بهترین نقطه‌ی عملکرد مدل بیشتر شود.

### چرا یک CFG مشترک برای هر دو مدل استفاده شد؟

برای مقایسه‌ی منصفانه بین **GCN** و **GraphSAGE**، هایپرپارامترها را یکسان نگه داشتیم تا تغییرات عملکرد به اختلاف معماری‌ها نسبت داده شود، نه به این‌که یک مدل با تنظیمات بهتری آموزش دیده باشد. به همین دلیل، هر دو مدل با همان hidden_dim، dropout، lr، weight_decay و epochs آموزش داده شدند.

## جمع‌بندی

به طور خلاصه، این تنظیمات یک پیکربندی پایه و قابل دفاع برای اجرای مدل‌های گرافی روی دیتاست‌های خانواده‌ی Planetoid است: دو لایه برای استفاده از همسایه‌های تا فاصله‌ی دو، بعد پنهان متوسط برای تعادل بین ظرفیت و overfitting، dropout و weight decay برای regularization در شرایط داده‌ی برچسب‌دار محدود، و learning rate و epochs برای آموزش پایدار و قابل تکرار در مقایسه‌ی GCN و GraphSAGE.

</div>


In [5]:
import os, json
import pandas as pd
from torch_geometric.datasets import Planetoid

def mask_count(mask):
    return int(mask.sum().item())

for DATASET in DATASETS:
    ds = Planetoid(root=DATA_DIR, name=DATASET)
    data = ds[0].to(DEVICE)

    basic = {
        "dataset": DATASET,
        "num_nodes": int(data.num_nodes),
        "num_edges": int(data.num_edges),
        "num_features": int(ds.num_features),
        "num_classes": int(ds.num_classes),
        "train_nodes": mask_count(data.train_mask),
        "val_nodes": mask_count(data.val_mask),
        "test_nodes": mask_count(data.test_mask),
    }

    df_basic = pd.DataFrame([basic])
    print(df_basic)

    save_dir = os.path.join(RESULTS_DIR, DATASET.lower(), "dataset")
    os.makedirs(save_dir, exist_ok=True)

    df_basic.to_csv(os.path.join(save_dir, "basic_dataset_stats.csv"), index=False)
    with open(os.path.join(save_dir, "basic_dataset_stats.json"), "w", encoding="utf-8") as f:
        json.dump(basic, f, indent=2)

    print(f"Saved: {os.path.join(save_dir, 'basic_dataset_stats.csv')}")
    print(f"Saved: {os.path.join(save_dir, 'basic_dataset_stats.json')}")


  dataset  num_nodes  num_edges  num_features  num_classes  train_nodes  \
0    Cora       2708      10556          1433            7          140   

   val_nodes  test_nodes  
0        500        1000  
Saved: /home/amir/Desktop/proposal/results/cora/dataset/basic_dataset_stats.csv
Saved: /home/amir/Desktop/proposal/results/cora/dataset/basic_dataset_stats.json
  dataset  num_nodes  num_edges  num_features  num_classes  train_nodes  \
0  PubMed      19717      88648           500            3           60   

   val_nodes  test_nodes  
0        500        1000  
Saved: /home/amir/Desktop/proposal/results/pubmed/dataset/basic_dataset_stats.csv
Saved: /home/amir/Desktop/proposal/results/pubmed/dataset/basic_dataset_stats.json


<div dir="rtl" align="right">

## شباهت ساختاری CiteSeer، Cora و PubMed و دلیل انتقال مدل بین دیتاست‌ها (با اتکا به آمار استخراج‌شده)

از آن‌جایی که مدل اولیه‌ی ما روی **CiteSeer** طراحی/پیاده‌سازی شده بود، برای اجرای همان معماری روی **Cora** و **PubMed** لازم بود نشان بدهیم که این دیتاست‌ها از نظر **ماهیت گراف و ویژگی‌های ساختاری** در یک خانواده قرار می‌گیرند. هر سه دیتاست، **گراف استنادی** هستند (گره=مقاله، یال=ارجاع) و همین نوع گراف‌ها معمولاً الگوهای مشابهی در اتصال‌پذیری و توزیع درجه دارند؛ بنابراین انتظار داریم مدل‌های پیام‌رسانی مانند GCN و GraphSAGE بتوانند روی هر سه دیتاست رفتار قابل مقایسه‌ای نشان دهند.

اولین نکته‌ی روشن در آمارهای ما، **پراکنده بودن (sparse) هر سه گراف** است. چگالی گراف‌ها بسیار پایین است: برای PubMed حدود **0.00023**، برای CiteSeer حدود **0.00082** و برای Cora حدود **0.00144**. این اعداد یعنی نسبت یال به تعداد کل زوج‌گره‌های ممکن بسیار کوچک است و ساختار شبکه‌ها از نظر “کم‌تراکم بودن” مشابه است. در چنین شرایطی معماری‌های چندلایه‌ی سبک (مثل دو لایه کانولوشن) معمولاً انتخاب مناسبی هستند و مدل بدون نیاز به پیچیده‌سازی شدید می‌تواند از همسایگی‌ها سیگنال بگیرد.

دوم، **میانگین درجه‌ها** هم در یک بازه‌ی نزدیک قرار دارند و نشان می‌دهد این شبکه‌ها از نظر میزان اتصال متوسطِ هر گره، رفتار هم‌خانواده دارند: PubMed با میانگین درجه‌ی حدود **4.50**، Cora حدود **3.90** و CiteSeer حدود **2.74**. یعنی با اینکه اندازه‌ی گراف‌ها متفاوت است (PubMed بزرگ‌تر است)، اما “حجم متوسط همسایگی” در هر سه دیتاست در حد چند همسایه است؛ این دقیقاً همان حالتی است که پیام‌رسانی محلی در GNNها موثر می‌شود.

سوم، از نظر **ساختار مسیرها** هم شباهت قابل توجهی دیده می‌شود. میانگین طول کوتاه‌ترین مسیر در PubMed حدود **6.34** و در Cora حدود **6.31** است که تقریباً یکسان‌اند. این شباهت نشان می‌دهد با وجود تفاوت اندازه، ساختار کلی انتشار اطلاعات و “فاصله‌ی متوسط” در این دو شبکه نزدیک است. در CiteSeer این مقدار بزرگ‌تر است (حدود **9.32**) که با توجه به تعداد مؤلفه‌های هم‌بند بسیار بیشتر (حدود **438**) و کوچک‌تر بودن نسبت مؤلفه‌ی هم‌بند بزرگ (LCC) قابل توجیه است؛ اما همچنان CiteSeer در همان کلاس شبکه‌های استنادی پراکنده قرار می‌گیرد و از نظر جنس ساختار، فاصله‌ی کیفی زیادی با Cora/PubMed ندارد.

چهارم، **الگوی اتصال‌پذیری و مؤلفه‌های هم‌بند** کمک می‌کند تفاوت‌ها و شباهت‌ها را دقیق‌تر ببینیم. PubMed عملاً یک گراف یکپارچه است (تعداد مؤلفه‌ها **1** و نسبت LCC برابر **1.0**). Cora چند مؤلفه دارد (حدود **78**) اما همچنان بخش عمده‌ی گراف در LCC قرار دارد (نسبت LCC حدود **0.918**). CiteSeer تکه‌تکه‌تر است (حدود **438** مؤلفه و نسبت LCC حدود **0.637**)؛ این یعنی CiteSeer از نظر اتصال‌پذیری کلی ضعیف‌تر است، ولی همچنان یک LCC قابل توجه دارد و تحلیل‌ها/مدل‌سازی روی همان هسته‌ی اصلی معنی‌دار می‌ماند. نتیجه‌ی عملی این مشاهده این است که مدلِ ساخته‌شده برای CiteSeer وقتی روی Cora و PubMed منتقل می‌شود، در واقع روی گراف‌هایی اجرا می‌شود که حتی از نظر اتصال‌پذیری “بهتر” یا “کم‌تکه‌تر” هستند؛ بنابراین طبیعی است که عملکرد مدل روی آن‌ها افت نکند و حتی پایدارتر باشد.

پنجم، برای اینکه نشان بدهیم این گراف‌ها از نظر شکل شبکه شبیه‌اند، به **توزیع درجه و دم‌سنگین بودن** نگاه می‌کنیم. بیشینه‌ی درجه در PubMed **171** و در Cora **168** است (تقریباً هم‌اندازه)، که نشان می‌دهد در هر دو شبکه، چند گره‌ی بسیار پرارتباط وجود دارد. CiteSeer هم چنین رفتاری دارد هرچند سقف درجه‌اش پایین‌تر است (حداکثر درجه **99**). همین وجود “گره‌های پرتعدادِ کم‌درجه در کنار چند گره‌ی بسیار پُردرجه” الگوی رایج گراف‌های استنادی است و با رسم نمودار توزیع درجه در مقیاس لگاریتمی می‌توان دم‌سنگین بودن را به‌صورت بصری نشان داد. به همین دلیل در سلول بعدی نمودارهای درجه‌ی هر سه دیتاست را رسم می‌کنیم تا مشابهت ساختاری به شکل تصویری هم قابل دفاع باشد.

جمع‌بندی اینکه: با توجه به پراکندگی بسیار بالا (چگالی‌های بسیار کوچک)، میانگین درجه‌های هم‌خانواده، وجود LCC قابل توجه، و نشانه‌های واضح از دم‌سنگین بودن توزیع درجه (خصوصاً نزدیکی degree_max در Cora و PubMed)، می‌توان استدلال کرد که CiteSeer، Cora و PubMed از نظر ساختار گراف در یک خانواده قرار می‌گیرند. بنابراین اجرای همان مدلِ پیاده‌سازی‌شده روی CiteSeer برای Cora و PubMed انتخاب منطقی و قابل دفاعی است.

</div>


In [6]:
import os, json
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_networkx

def exact_avg_shortest_path_length(G):
    nodes = list(G.nodes())
    if len(nodes) < 2:
        return None
    total = 0
    cnt = 0
    for s in nodes:
        d = nx.single_source_shortest_path_length(G, s)
        total += sum(d.values())
        cnt += (len(d) - 1)
    return (total / cnt) if cnt > 0 else None

for DATASET in ["Cora", "CiteSeer", "PubMed"]:
    ds_local = Planetoid(root=DATA_DIR, name=DATASET)
    data_local = ds_local[0].to("cpu")
    G = to_networkx(data_local, to_undirected=True)

    n = G.number_of_nodes()
    m = G.number_of_edges()
    avg_degree = (2 * m / n) if n > 0 else 0.0
    density = nx.density(G) if n > 1 else 0.0
    num_cc = nx.number_connected_components(G) if n > 0 else 0

    lcc_nodes = max(nx.connected_components(G), key=len) if n > 0 else []
    LCC = G.subgraph(lcc_nodes).copy()
    lcc_size = LCC.number_of_nodes()
    lcc_ratio = (lcc_size / n) if n > 0 else 0.0

    avg_clustering = nx.average_clustering(G) if n > 1 else 0.0
    asp = exact_avg_shortest_path_length(G)
    asp_lcc = exact_avg_shortest_path_length(LCC)

    degrees = [d for _, d in G.degree()]
    plt.figure()
    plt.hist(degrees, bins=50)
    plt.yscale("log")
    plt.xlabel("Degree")
    plt.ylabel("Count (log)")
    plt.title(f"Degree Distribution - {DATASET}")
    plt.tight_layout()

    save_dir = os.path.join(RESULTS_DIR, DATASET.lower(), "structure")
    os.makedirs(save_dir, exist_ok=True)

    fig_path = os.path.join(save_dir, f"degree_dist_{DATASET.lower()}.png")
    plt.savefig(fig_path, dpi=200)
    plt.close()

    struct = {
        "dataset": DATASET,
        "num_nodes": int(n),
        "num_edges_undirected": int(m),
        "avg_degree": float(avg_degree),
        "density": float(density),
        "num_connected_components": int(num_cc),
        "lcc_nodes": int(lcc_size),
        "lcc_ratio": float(lcc_ratio),
        "avg_clustering": float(avg_clustering),
        "avg_shortest_path_len": float(asp) if asp is not None else None,
        "avg_shortest_path_len_lcc": float(asp_lcc) if asp_lcc is not None else None,
        "degree_min": int(min(degrees)) if degrees else 0,
        "degree_max": int(max(degrees)) if degrees else 0,
    }

    print(struct)

    with open(os.path.join(save_dir, "structural_stats.json"), "w", encoding="utf-8") as f:
        json.dump(struct, f, indent=2)

    print("Saved:", os.path.join(save_dir, "structural_stats.json"))
    print("Saved:", fig_path)


{'dataset': 'Cora', 'num_nodes': 2708, 'num_edges_undirected': 5278, 'avg_degree': 3.8980797636632203, 'density': 0.0014399999126942077, 'num_connected_components': 78, 'lcc_nodes': 2485, 'lcc_ratio': 0.9176514032496307, 'avg_clustering': 0.24067329850193736, 'avg_shortest_path_len': 6.310310801906627, 'avg_shortest_path_len_lcc': 6.310998681298742, 'degree_min': 1, 'degree_max': 168}
Saved: /home/amir/Desktop/proposal/results/cora/structure/structural_stats.json
Saved: /home/amir/Desktop/proposal/results/cora/structure/degree_dist_cora.png
{'dataset': 'CiteSeer', 'num_nodes': 3327, 'num_edges_undirected': 4552, 'avg_degree': 2.7363991584009617, 'density': 0.0008227297529768376, 'num_connected_components': 438, 'lcc_nodes': 2120, 'lcc_ratio': 0.6372107003306282, 'avg_clustering': 0.14147102442629098, 'avg_shortest_path_len': 9.32317763436192, 'avg_shortest_path_len_lcc': 9.329714532486845, 'degree_min': 0, 'degree_max': 99}
Saved: /home/amir/Desktop/proposal/results/citeseer/structure/

<div dir="rtl" style="text-align:right; font-family: Vazirmatn, IRANSans, Tahoma, Arial; line-height:1.9;">

## مقایسه‌ی مدل‌های GCN و GraphSAGE در پروژه

در این پروژه برای مسئله‌ی طبقه‌بندی گره‌ها روی دیتاست‌های **Cora** و **PubMed**، دو مدل **GCN** و **GraphSAGE** پیاده‌سازی و با هم مقایسه شدند. هر دو مدل از دو لایه‌ی پیام‌رسانی روی گراف استفاده می‌کنند و سپس یک لایه‌ی خطی قرار دارد تا بردار نهفته‌ی هر گره را به کلاس‌های خروجی تبدیل کند. همچنین در هر دو مدل از تابع فعال‌ساز ReLU و dropout برای کاهش overfitting استفاده شده است. برای اینکه مقایسه منصفانه باشد، تنظیمات آموزشی (هایپرپارامترها، تعداد epoch، بهینه‌ساز و …) برای هر دو مدل یکسان در نظر گرفته شده است.

### مدل GCN

GCN (Graph Convolutional Network) از ایده‌ی هموارسازی ویژگی‌ها روی گراف استفاده می‌کند؛ یعنی نمایش هر گره با ترکیب اطلاعات همسایه‌ها ساخته می‌شود تا ویژگی‌های محلی گراف بهتر در نمایش نهایی منعکس شود. در پیاده‌سازی ما، دو لایه‌ی GCNConv داریم که باعث می‌شود هر گره اطلاعات همسایه‌های نزدیک‌تر را دریافت کند و سپس یک لایه‌ی خطی برای پیش‌بینی کلاس‌ها اعمال می‌شود. معمولاً GCN در گراف‌های استنادی و سناریوهای نیمه‌نظارتی کلاسیک، عملکرد خوبی دارد چون ساختار گراف و شباهت همسایه‌ها (از نظر موضوع/برچسب) به مدل کمک می‌کند.

### مدل GraphSAGE

GraphSAGE نیز یک روش مبتنی بر پیام‌رسانی است، اما ایده‌ی اصلی آن استفاده از یک سازوکار تجمیع (aggregation) برای جمع‌کردن اطلاعات همسایه‌ها و ساخت نمایش جدید است. در کد ما از دو لایه‌ی SAGEConv استفاده شده و مانند GCN بعد از دو لایه، یک لایه‌ی خطی برای خروجی داریم. GraphSAGE به‌طور کلی برای مقیاس‌های بزرگ‌تر و سناریوهایی که تعمیم‌پذیری مهم است (مثلاً یادگیری نمایش‌هایی که روی گره‌های جدید هم قابل استفاده باشند) شناخته می‌شود و از نظر نوع تجمیع همسایه‌ها انعطاف بیشتری دارد.

### جمع‌بندی تفاوت‌ها

به طور کلی، GCN بیشتر بر هموارسازی و استفاده‌ی مستقیم از ساختار گراف تکیه دارد و در بسیاری از مسائل کلاسیک گراف‌های استنادی نتایج بسیار رقابتی می‌دهد. در مقابل، GraphSAGE علاوه بر دقت، از نظر طراحی برای تعمیم‌پذیری و مقیاس‌پذیری هم مطرح است. به همین دلیل در این پروژه هر دو مدل روی دو دیتاست اجرا شدند تا هم یک مدل پایه‌ی قوی (GCN) داشته باشیم و هم یک روش دیگر (GraphSAGE) را از نظر عملکرد، پایداری و رفتار روی داده مقایسه کنیم.

</div>


<div dir="rtl" style="text-align:right; font-family: Vazirmatn, IRANSans, Tahoma, Arial; line-height:1.9;">

## چرا برای این پروژه سراغ GCN و GraphSAGE رفتیم؟

برای انتخاب مدل‌ها یک دلیل محکم وجود دارد: دیتاست‌های ما (به‌خصوص **Cora** و **PubMed**) دقیقاً از آن جنس داده‌هایی هستند که «اطلاعات گرافی» نقش اصلی در یادگیری دارد، نه یک ویژگی تزئینی. هر دو دیتاست **شبکه‌ی استنادی** هستند؛ یعنی هر گره یک مقاله است، یال‌ها رابطه‌ی استناد را نشان می‌دهند و برچسب‌ها موضوع/حوزه‌ی مقاله‌ها هستند. در چنین شبکه‌هایی معمولاً مقاله‌هایی که به هم استناد می‌کنند یا در ناحیه‌های نزدیک گراف قرار می‌گیرند، از نظر موضوعی نیز شباهت دارند. بنابراین اگر فقط ویژگی‌های متنی/برداری گره‌ها را در نظر بگیریم، بخش مهمی از سیگنال را از دست می‌دهیم؛ چون خودِ «همسایه‌های گرافی» یک مقاله، سرنخ قوی برای تشخیص موضوع آن مقاله است. مدل‌های GNN دقیقاً برای همین طراحی شده‌اند: ترکیبِ ویژگی‌های گره با ساختار همسایگی از طریق پیام‌رسانی روی گراف.

نکته‌ی دوم این است که این مسئله به‌صورت کلاسیک **نیمه‌نظارتی** است؛ یعنی تعداد برچسب‌های آموزشی محدود است و مدل باید از ساختار گراف کمک بگیرد تا بهتر تعمیم بدهد. مدل‌هایی مثل MLP که گراف را کنار می‌گذارند، در این شرایط معمولاً یا overfit می‌شوند یا قدرت تعمیم‌شان کمتر می‌شود، چون فقط به ویژگی‌ها تکیه می‌کنند. در مقابل، GCN و GraphSAGE با پیام‌رسانی (message passing) عملاً از اطلاعات موجود در همسایگی و بخش بزرگی از گراف بهره می‌برند و این دقیقاً همان چیزی است که در دیتاست‌های استنادی به کار می‌آید.

دلیل سوم این است که انتخاب این دو مدل در کنار هم یک مقایسه‌ی قابل دفاع و علمی ایجاد می‌کند. **GCN** یک baseline بسیار استاندارد و شناخته‌شده برای طبقه‌بندی گره‌ها در گراف‌های استنادی است و معمولاً انتظار می‌رود حداقل به‌عنوان خط پایه گزارش شود. در کنار آن، **GraphSAGE** دیدگاهی نزدیک اما متفاوت ارائه می‌دهد: به جای تکیه‌ی کامل بر فرم کانولوشن گرافی کلاسیک، روی ایده‌ی «تجمیع همسایه‌ها» (aggregation) استوار است و از نظر طراحی، بیشتر با مفاهیمی مثل تعمیم‌پذیری و مقیاس‌پذیری شناخته می‌شود. بنابراین وقتی هر دو را کنار هم اجرا می‌کنیم، فقط نمی‌گوییم “یک مدل جواب داد”، بلکه دو رویکرد رایج و مهم از خانواده‌ی GNNها را روی مسئله‌ای مناسب محک می‌زنیم؛ در نتیجه انتخاب مدل‌ها اتفاقی نیست و بر اساس ماهیت داده و هدف مقایسه انجام شده است.

در جمع‌بندی، دو ویژگی کلیدی در این پروژه وجود دارد: **ساختار گرافی معنادار** و **برچسب‌های آموزشی محدود**. این دو ویژگی مستقیماً نشان می‌دهند که باید سراغ مدل‌های مبتنی بر گراف برویم. در میان گزینه‌های رایج، GCN به‌عنوان baseline استاندارد و GraphSAGE به‌عنوان یک روش پیام‌رسانی/تجمیعی معتبر، انتخاب‌هایی هستند که هم علمی و قابل دفاع‌اند و هم خروجی‌های قابل اتکا و مقایسه‌پذیر تولید می‌کنند.

</div>


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv

class GCNNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        emb = x
        out = self.lin(emb)
        return out, emb

class GraphSAGENet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        emb = x
        out = self.lin(emb)
        return out, emb


In [8]:
import os, time, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def train_one_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out, _ = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss.item())

@torch.no_grad()
def eval_all(model, data):
    model.eval()
    logits, emb = model(data.x, data.edge_index)
    preds = logits.argmax(dim=1).cpu().numpy()
    y = data.y.cpu().numpy()

    def masked(mask):
        idx = mask.cpu().numpy()
        y_true = y[idx]
        y_pred = preds[idx]
        return {
            "acc": float(accuracy_score(y_true, y_pred)),
            "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1_micro": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
            "prec_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "rec_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        }

    return masked(data.train_mask), masked(data.val_mask), masked(data.test_mask), preds, emb.detach().cpu().numpy()

def save_confusion_matrix(y_true, y_pred, num_classes, title, path):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1

    plt.figure(figsize=(6,5))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Pred")
    plt.ylabel("True")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    return cm.tolist()

def run_experiment(model_name, model_ctor):
    in_dim = ds.num_features
    out_dim = ds.num_classes

    save_dir = os.path.join(RESULTS_DIR, DATASET.lower(), model_name.lower())
    os.makedirs(save_dir, exist_ok=True)

    model = model_ctor(in_dim, CFG["hidden_dim"], out_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    criterion = torch.nn.CrossEntropyLoss()

    history = {"epoch": [], "train_loss": [], "val_acc": []}
    best_val = -1.0
    best_state = None

    t0 = time.time()
    for epoch in range(1, CFG["epochs"] + 1):
        loss = train_one_epoch(model, data, optimizer, criterion)
        train_m, val_m, test_m, preds, emb = eval_all(model, data)

        history["epoch"].append(epoch)
        history["train_loss"].append(loss)
        history["val_acc"].append(val_m["acc"])

        if val_m["acc"] >= best_val:
            best_val = val_m["acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
        model.eval()

    train_m, val_m, test_m, preds, emb = eval_all(model, data)
    runtime_s = float(time.time() - t0)

    metrics = {
        "dataset": DATASET,
        "model": model_name,
        "cfg": CFG,
        "best_val_acc": float(best_val),
        "runtime_s": runtime_s,
        "train": train_m,
        "val": val_m,
        "test": test_m,
    }

    with open(os.path.join(save_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    with open(os.path.join(save_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    np.save(os.path.join(save_dir, "preds.npy"), preds)
    np.save(os.path.join(save_dir, "embeddings.npy"), emb)

    plt.figure()
    plt.plot(history["epoch"], history["train_loss"])
    plt.xlabel("Epoch")
    plt.ylabel("Train Loss")
    plt.title(f"{model_name} - {DATASET} - Train Loss")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "train_loss.png"), dpi=200)
    plt.close()

    plt.figure()
    plt.plot(history["epoch"], history["val_acc"])
    plt.xlabel("Epoch")
    plt.ylabel("Val Acc")
    plt.title(f"{model_name} - {DATASET} - Val Acc")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "val_acc.png"), dpi=200)
    plt.close()

    y_true_test = data.y[data.test_mask].cpu().numpy()
    y_pred_test = preds[data.test_mask.cpu().numpy()]
    cm = save_confusion_matrix(
        y_true_test, y_pred_test, out_dim,
        title=f"Confusion Matrix (Test) - {model_name} - {DATASET}",
        path=os.path.join(save_dir, "confusion_matrix.png")
    )

    with open(os.path.join(save_dir, "confusion_matrix.json"), "w", encoding="utf-8") as f:
        json.dump(cm, f, indent=2)

    return metrics


<div dir="rtl" style="text-align:right; font-family: Vazirmatn, IRANSans, Tahoma, Arial; line-height:1.9;">

## معیارهای ارزیابی مدل

در این پروژه مسئله‌ی ما «طبقه‌بندی گره‌ها» روی دیتاست‌های خانواده‌ی Planetoid است (در این نوت‌بوک روی **Cora** و **PubMed** آموزش و ارزیابی انجام شده). برای ارزیابی مدل، به یک معیار اکتفا نکردیم؛ چون یک معیار مثل Accuracy فقط یک دید کلی می‌دهد و همیشه تمام جنبه‌های عملکرد مدل را نشان نمی‌دهد، مخصوصاً وقتی عملکرد مدل روی بعضی کلاس‌ها ضعیف‌تر باشد. به همین دلیل چند معیار را هم‌زمان گزارش می‌کنیم تا هم کیفیت کلی مشخص شود و هم رفتار مدل روی کلاس‌ها بهتر قابل تحلیل باشد.

**Accuracy** نشان می‌دهد چند درصد نمونه‌ها درست پیش‌بینی شده‌اند و یک معیار ساده و سریع برای مقایسه‌ی اولیه‌ی دو مدل (GCN و GraphSAGE) است. با این حال، اگر بعضی کلاس‌ها سخت‌تر یا کم‌نمونه‌تر باشند، ممکن است Accuracy تصویر کامل و منصفانه‌ای از عملکرد ندهد. برای همین از معیارهای مبتنی بر Precision/Recall و F1 هم استفاده می‌کنیم.

**Precision (ماکرو)** بیان می‌کند وقتی مدل یک کلاس را پیش‌بینی می‌کند، چند درصد از این پیش‌بینی‌ها واقعاً درست بوده‌اند. این معیار کمک می‌کند بفهمیم مدل آیا بیش از حد به سمت یک کلاس تمایل پیدا کرده و نمونه‌های دیگر را اشتباه به آن نسبت می‌دهد یا نه. **Recall (ماکرو)** از زاویه‌ی مقابل نگاه می‌کند: از بین نمونه‌های واقعی هر کلاس، چه سهمی را مدل درست پیدا کرده است. هر دو معیار را در حالت ماکرو گزارش می‌کنیم تا هر کلاس وزن برابر داشته باشد و کلاس‌های کوچک‌تر نادیده گرفته نشوند.

**F1-Score** معیار ترکیبی Precision و Recall است و وقتی مفید است که بخواهیم تعادل بین این دو را ببینیم. ما هم **Macro-F1** و هم **Micro-F1** را گزارش می‌کنیم. Macro-F1 به هر کلاس وزن برابر می‌دهد و اگر مدل روی کلاس‌های کم‌نمونه بد عمل کند، سریع‌تر مشخص می‌شود. Micro-F1 بیشتر تصویر کلی روی کل نمونه‌ها را نشان می‌دهد و معمولاً تحت تأثیر کلاس‌های پرتعدادتر است. داشتن هر دو معیار باعث می‌شود هم یک دید کلی داشته باشیم و هم مطمئن شویم کیفیت مدل قربانی چند کلاس پرتعداد نشده است.

علاوه بر معیارهای عددی، **Confusion Matrix** را هم ذخیره می‌کنیم تا دقیقاً مشخص شود مدل بیشتر بین کدام کلاس‌ها اشتباه می‌کند. این بخش برای تحلیل کیفی بسیار مهم است، چون با دیدن صرفِ Accuracy یا F1 متوجه نمی‌شویم خطاها از چه نوعی هستند و چه کلاس‌هایی بیشتر با هم اشتباه گرفته می‌شوند.

در طول آموزش نیز **Best Validation Accuracy** را نگه می‌داریم تا بهترین نقطه‌ی عملکرد مدل در فرآیند آموزش انتخاب شود و صرفاً به epoch آخر وابسته نباشیم (چون ممکن است مدل در انتهای آموزش overfit کند). همچنین **Runtime** (زمان اجرا) را ثبت می‌کنیم چون در عمل، علاوه بر کیفیت، هزینه‌ی زمانی هم اهمیت دارد و ممکن است یک مدل با اختلاف ناچیزِ دقت، زمان بیشتری مصرف کند.

در نهایت، برای تصمیم‌گیری و مقایسه‌ی اصلی بین مدل‌ها، ما بیشترین وزن را به **Test Accuracy** و به‌خصوص **Macro-F1** می‌دهیم (به دلیل منصفانه‌تر بودن نسبت به کلاس‌ها). سایر معیارها و confusion matrix هم برای تحلیل تکمیلی و توضیح رفتار مدل استفاده می‌شوند.

</div>


In [9]:
import os
import pandas as pd
from torch_geometric.datasets import Planetoid

def ctor_gcn(in_dim, hidden_dim, out_dim):
    return GCNNet(in_dim, hidden_dim, out_dim, dropout=CFG["dropout"])

def ctor_sage(in_dim, hidden_dim, out_dim):
    return GraphSAGENet(in_dim, hidden_dim, out_dim, dropout=CFG["dropout"])

all_rows = []

for DATASET in ["Cora", "PubMed"]:
    ds = Planetoid(root=DATA_DIR, name=DATASET)
    data = ds[0].to(DEVICE)

    m_gcn = run_experiment("GCN", ctor_gcn)
    m_sage = run_experiment("GraphSAGE", ctor_sage)

    rows = []
    for m in [m_gcn, m_sage]:
        rows.append({
            "dataset": m["dataset"],
            "model": m["model"],
            "test_acc": m["test"]["acc"],
            "test_f1_micro": m["test"]["f1_micro"],
            "test_f1_macro": m["test"]["f1_macro"],
            "test_prec_macro": m["test"]["prec_macro"],
            "test_rec_macro": m["test"]["rec_macro"],
            "runtime_s": m["runtime_s"],
            "best_val_acc": m["best_val_acc"],
        })
        all_rows.append(rows[-1])

    df_results = pd.DataFrame(rows).sort_values(["model"])
    print("\n", DATASET)
    print(df_results)

    save_dir = os.path.join(RESULTS_DIR, DATASET.lower(), "summary")
    os.makedirs(save_dir, exist_ok=True)

    out_csv = os.path.join(save_dir, f"preliminary_results_{DATASET.lower()}.csv")
    df_results.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

df_all = pd.DataFrame(all_rows).sort_values(["dataset", "model"])
print("\nALL DATASETS")
print(df_all)

out_all = os.path.join(RESULTS_DIR, "preliminary_results_all.csv")
df_all.to_csv(out_all, index=False)
print(f"Saved: {out_all}")



 Cora
  dataset      model  test_acc  test_f1_micro  test_f1_macro  test_prec_macro  \
0    Cora        GCN     0.815          0.815       0.803798         0.791151   
1    Cora  GraphSAGE     0.794          0.794       0.782090         0.779201   

   test_rec_macro  runtime_s  best_val_acc  
0        0.823261   5.096297         0.812  
1        0.794287  11.805414         0.800  
Saved: /home/amir/Desktop/proposal/results/cora/summary/preliminary_results_cora.csv

 PubMed
  dataset      model  test_acc  test_f1_micro  test_f1_macro  test_prec_macro  \
0  PubMed        GCN     0.769          0.769       0.767475         0.764832   
1  PubMed  GraphSAGE     0.763          0.763       0.759969         0.757511   

   test_rec_macro  runtime_s  best_val_acc  
0        0.770734  21.927711         0.796  
1        0.762626  31.004683         0.800  
Saved: /home/amir/Desktop/proposal/results/pubmed/summary/preliminary_results_pubmed.csv

ALL DATASETS
  dataset      model  test_acc  test_f